In [0]:
t0 = time.time()
inferred = spark.read.option("multiLine", True).json(RT)
n = inferred.count()
infer_secs = time.time() - t0
print(f"inferred read: {infer_secs:.1f}s · {n:,} files")

inferred.printSchema()

In [0]:
from pyspark.sql.types import (StructType, StructField, ArrayType, StringType,
                               DoubleType, LongType, IntegerType, BooleanType)

carriage = StructType([
    StructField("carriage_sequence",    IntegerType()),
    StructField("label",                StringType()),
    StructField("occupancy_status",     StringType()),
    StructField("occupancy_percentage", IntegerType()),
    StructField("orientation",          StringType()),
])

vehicle = StructType([
    StructField("current_status",        StringType()),
    StructField("current_stop_sequence", IntegerType()),
    StructField("timestamp",             LongType()),
    StructField("occupancy_status",      StringType()),
    StructField("occupancy_percentage",  IntegerType()),
    StructField("stop_id",               StringType()),
    StructField("position", StructType([
        StructField("latitude",  DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("bearing",   DoubleType()),
        StructField("speed",     DoubleType()),
    ])),
    StructField("trip", StructType([
        StructField("route_id",              StringType()),
        StructField("trip_id",               StringType()),
        StructField("direction_id",          IntegerType()),
        StructField("start_date",            StringType()),
        StructField("start_time",            StringType()),
        StructField("schedule_relationship", StringType()),
        StructField("last_trip",             BooleanType()),
        StructField("revenue",               BooleanType()),
    ])),
    StructField("vehicle", StructType([
        StructField("id",    StringType()),
        StructField("label", StringType()),
    ])),
    StructField("multi_carriage_details", ArrayType(carriage)),
])

FEED_SCHEMA = StructType([
    StructField("header", StructType([
        StructField("gtfs_realtime_version", StringType()),
        StructField("incrementality",        StringType()),
        StructField("timestamp",             LongType()),
    ])),
    StructField("entity", ArrayType(StructType([
        StructField("id",      StringType()),
        StructField("vehicle", vehicle),
    ]))),
])

print("FEED_SCHEMA defined")

In [0]:
def leaf_names(schema, prefix=""):
    """Every leaf field path. Structs recursed, arrays-of-struct marked []."""
    out = set()
    for f in schema.fields:
        p, dt = f"{prefix}{f.name}", f.dataType
        if hasattr(dt, "fields"):                                      # struct
            out |= leaf_names(dt, p + ".")
        elif hasattr(dt, "elementType") and hasattr(dt.elementType, "fields"):
            out |= leaf_names(dt.elementType, p + "[].")                # array of struct
        else:
            out.add(p)
    return out

print(len(leaf_names(FEED_SCHEMA)), "leaf fields declared")
for p in sorted(leaf_names(FEED_SCHEMA)):
    print(" ", p)

In [0]:
declared = leaf_names(FEED_SCHEMA)
actual   = leaf_names(inferred.schema)

missing = sorted(actual - declared - {"dt"})   # dt comes from partition discovery, not the feed
extra   = sorted(declared - actual)

print("IN THE DATA BUT NOT DECLARED (silently dropped):", missing or "none")
print("DECLARED BUT NOT IN THE DATA (will be null):    ", extra or "none")

assert not missing, f"schema is discarding real fields: {missing}"

In [0]:
def load_rt(path=RT):
    """Raw snapshots -> bronze shape. One row per file per entity.
    The entity struct stays intact; flattening is a silver concern."""
    return (spark.read
              .option("multiLine", True)
              .schema(FEED_SCHEMA)
              .json(path)
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_snapshot_ts", F.col("header.timestamp"))
            .withColumn("_dt", F.to_date(F.from_unixtime(F.col("header.timestamp"))))
            .select("_source_file", "_ingested_at", "_snapshot_ts", "_dt",
                    F.explode("entity").alias("entity")))

t0 = time.time()
(load_rt().write
   .format("delta")
   .mode("overwrite")
   .option("replaceWhere", "_dt IS NOT NULL")
   .saveAsTable("transit.bronze.rt_vehicle_positions"))
reload_secs = time.time() - t0

rt = spark.table("transit.bronze.rt_vehicle_positions")
n_files = rt.select("_source_file").distinct().count()
print(f"full reload: {reload_secs:.0f}s · {rt.count():,} rows · {n_files:,} files")

In [0]:
dupes = (spark.table("transit.bronze.rt_vehicle_positions")
           .groupBy("_source_file", "entity.id").count()
           .filter(F.col("count") > 1).count())
print("duplicate (file, entity) pairs:", dupes)
assert dupes == 0, "bronze RT contains duplicate rows"

In [0]:
spark.sql("""
  SELECT _dt,
         date_format(_dt, 'EEE')                        AS day,
         count(DISTINCT _source_file)                   AS snapshots,
         count(*)                                       AS rows,
         round(count(*) / count(DISTINCT _source_file)) AS per_snapshot
  FROM transit.bronze.rt_vehicle_positions
  GROUP BY _dt ORDER BY _dt
""").display()

In [0]:
rt = spark.table("transit.bronze.rt_vehicle_positions")
(rt.select(
    F.count("*").alias("total"),
    F.sum(F.col("entity.vehicle.trip.revenue").cast("int")).alias("revenue_true"),
    F.sum((~F.col("entity.vehicle.trip.revenue")).cast("int")).alias("revenue_false"),
    F.sum(F.col("entity.vehicle.trip.revenue").isNull().cast("int")).alias("revenue_null"))
 .display())